In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
import xgboost as xgb

End-to-end pipeline:
1. Data preparation from transaction history
2. Feature engineering
3. Hyperparameter tuning (RandomizedSearchCV)
4. Train multiple models (Linear Regression, Random Forest, Gradient Boosting, XGBoost)
5. Compare all models
6. Make predictions on new customers

Based on/Inspired by:
- Sequence Analysis paper: Sequential purchase patterns
- NCF-CI paper: Customer context information

In [11]:
# Loading data
def load_and_merge_data(customers_path, orders_path, products_path):
    """
    Load and merge the three separate datasets.
    
    Args:
        customers_path: Path to clean_customers.csv
        orders_path: Path to clean_orders.csv
        products_path: Path to clean_products.csv
    
    Returns:
        df2: Merged and aggregated dataframe ready for pipeline
    """
    
    # Load datasets
    print("\nLoading datasets")
    customers = pd.read_csv(customers_path)
    orders = pd.read_csv(orders_path)
    products = pd.read_csv(products_path)
    
    print(f"Customers: {len(customers):,} rows")
    print(f"Orders: {len(orders):,} rows (may include multiple items per order)")
    print(f"Products: {len(products):,} rows")
    
    # Merge orders + products to add category
    print("\nMerging orders with product categories")
    df = orders.merge(
        products[['product_code', 'category_main']],
        on='product_code',
        how='left'
    )
    
    # Rename columns to match pipeline expectations
    df = df.rename(columns={
        'category_main': 'category',
        'product_code': 'product_id'
    })
    
    print(f"Orders + Products merged: {len(df):,} rows")
    
    # Aggregate multiple items per order into single purchase
    print("\nAggregating multiple items per order...")
    df_aggregated = df.groupby(
        ['customer_id', 'order_id', 'purchase_date']
    ).agg({
        'line_total': 'sum', # Total order value
        'category': 'first', # Primary category (first item)
        'quantity': 'sum' if 'quantity' in df.columns else 'count'  # Total items
    }).reset_index()
    
    # Handle quantity column naming
    if 'quantity' not in df_aggregated.columns:
        if 'count' in df_aggregated.columns:
            df_aggregated = df_aggregated.rename(columns={df_aggregated.columns[-1]: 'quantity'})
    
    print(f"Original rows (with multiple items per order): {len(df):,}")
    print(f"Aggregated to unique orders: {len(df_aggregated):,}")
    
    # Merge customer context for additional features
    print("\nMerging with customer metrics...")
    df2 = df_aggregated.merge(
        customers[['customer_id', 'tenure_bucket', 'total_orders', 'has_returned']],
        on='customer_id',
        how='left'
    )
    
    print(f"Final merged dataset: {len(df2):,} rows")
    print(f"\nDataset ready for pipeline:")
    print(f"Columns: {df2.columns.tolist()}")
    print(f"Date range: {df2['purchase_date'].min()} to {df2['purchase_date'].max()}")
    print(f"Total customers: {df2['customer_id'].nunique():,}")
    print(f"Total unique orders: {len(df2):,}")
    print(f"Average items per order: {df2['quantity'].mean():.2f}")
    
    return df2

In [4]:
# Data Preparation
def prepare_next_purchase_data(df, rfm_context, min_purchases=2):
    """
    Transform transaction history into ML training dataset.
    - Handles aggregated orders (one row per order)
    - Multiple items per order already handled in load_and_merge_data()
    - Counts orders as purchases, not individual items
    
    Each row = "Given customer X's history up to date Y, when will they 
               purchase next and what category?"
    
    Args:
        df: Merged transaction dataframe from load_and_merge_data()
            Columns: [customer_id, order_id, purchase_date, line_total, category]
        rfm_context: RFM + clustering dataframe from previous notebook
        min_purchases: Minimum purchases required to create training example
    
    Returns:
        training_data: Dataframe with features for ML model
    """
    
    df = df.copy()
    df['purchase_date'] = pd.to_datetime(df['purchase_date'])
    
    # Sort by customer and date
    df = df.sort_values(['customer_id', 'purchase_date']).reset_index(drop=True)
    
    training_data = []
    skipped_customers = 0
    
    print(f"\nProcessing {df['customer_id'].nunique():,} customers...")
    
    # For each customer, create training examples from their purchase history
    for idx, customer_id in enumerate(df['customer_id'].unique()):
        if (idx + 1) % 2000 == 0:
            print(f"Processed {idx + 1:,} customers")
        
        customer_txns = df[df['customer_id'] == customer_id].copy()
        
        # Need at least 2 purchases to predict the next
        if len(customer_txns) < min_purchases:
            skipped_customers += 1
            continue
        
        # Get unique purchase dates (already one row per purchase now)
        purchase_dates = sorted(customer_txns['purchase_date'].unique())
        
        # For each purchase (except last), create a training example
        for i in range(len(purchase_dates) - 1):
            current_purchase_date = purchase_dates[i]
            next_purchase_date = purchase_dates[i + 1]
            
            # Historical data: everything before current purchase
            historical_txns = customer_txns[customer_txns['purchase_date'] < current_purchase_date]
            
            if len(historical_txns) == 0:
                continue
            
            # Next purchase data
            next_txns = customer_txns[customer_txns['purchase_date'] == next_purchase_date]
            
            # Calculate days until next purchase
            days_to_next = (next_purchase_date - current_purchase_date).days
            
            # Next purchase category
            next_category = next_txns['category'].iloc[0]
            
            # Historical features (RFM-like metrics from history only)
            # frequency now counts orders, not items
            hist_frequency = len(historical_txns)  # Number of orders
            hist_monetary = historical_txns['line_total'].sum()
            hist_avg_order = hist_monetary / hist_frequency if hist_frequency > 0 else 0
            hist_days_active = (historical_txns['purchase_date'].max() - historical_txns['purchase_date'].min()).days
            
            # Category affinity
            category_counts = historical_txns['category'].value_counts()
            top_category = category_counts.idxmax()
            category_concentration = category_counts.max() / len(historical_txns)
            
            # Get customer context from RFM
            customer_context = rfm_context[rfm_context['customer_id'] == customer_id]
            
            if customer_context.empty:
                continue
            
            example = {
                'customer_id': customer_id,
                'observation_date': current_purchase_date.date(),
                'next_purchase_date': next_purchase_date.date(),
                'days_to_next_purchase': days_to_next,
                'next_category': next_category,
                # Historical RFM features
                'hist_frequency': hist_frequency,
                'hist_monetary': hist_monetary,
                'hist_avg_order': hist_avg_order,
                'hist_days_active': hist_days_active,
                'category_concentration': category_concentration,
                'top_historical_category': top_category,
                # Customer context
                'rfm_segment': customer_context['segment'].values[0],
                'kmeans_cluster': customer_context['kmeans_cluster'].values[0],
                'rfm_score': customer_context['RFM_score'].values[0],
                'monetary': customer_context['monetary'].values[0],
                'recency_days': customer_context['recency_days'].values[0],
                'frequency': customer_context['frequency'].values[0],
            }
            
            training_data.append(example)
    
    training_df = pd.DataFrame(training_data)
    print(f"\nCreated {len(training_df):,} training examples from {df['customer_id'].nunique():,} customers.")    
    
    return training_df

In [5]:
def engineer_features(training_df):
    """
    Create ML features from raw training data.
    Encode categorical variables, scale numerical features.
    """

    df = training_df.copy()
    
    # Encode categorical variables
    le_segment = LabelEncoder()
    le_category = LabelEncoder()
    
    df['segment_encoded'] = le_segment.fit_transform(df['rfm_segment'])
    
    # Fit encoder on all possible categories
    # Get all unique categories from both target and feature columns
    all_categories = pd.concat([df['top_historical_category'], df['next_category']]).unique()
    
    le_category.fit(all_categories) # Fit on all possible values
    
    df['top_category_encoded'] = le_category.transform(df['top_historical_category'])
    df['next_category_encoded'] = le_category.transform(df['next_category'])
    
    # Feature list for ML models
    feature_cols = [
        'hist_frequency',
        'hist_monetary',
        'hist_avg_order',
        'hist_days_active',
        'category_concentration',
        'segment_encoded',
        'kmeans_cluster',
        'monetary',
        'recency_days',
        'frequency',
        'top_category_encoded'
    ]
    
    # Scale numerical features (except encoded categoricals)
    scaler = StandardScaler()
    numerical_cols = [
        'hist_frequency', 'hist_monetary', 'hist_avg_order', 'hist_days_active',
        'category_concentration', 'monetary', 'recency_days', 'frequency'
    ]
    
    df_scaled = df.copy()
    df_scaled[numerical_cols] = scaler.fit_transform(df[numerical_cols])
    
    print(f"\nFeature engineering complete")
    print(f"Total features: {len(feature_cols)}")
    print(f"Numerical features (scaled): {len(numerical_cols)}")
    print(f"Categorical features (encoded): {len(feature_cols) - len(numerical_cols)}")
    print(f"Features: {feature_cols}")
    print(f"\nCategories found: {len(all_categories)}")
    print(f"{','.join(sorted(all_categories))}")
    
    return df_scaled, feature_cols, scaler, le_segment, le_category

In [6]:
# Train/Test Split Function
def split_data_by_customer(df, feature_cols, test_size=0.2, random_state=42): # Follwing 80/20 split
    """
    Split data by customer (not random rows) to avoid leakage.
    """

    # Split by unique customers
    unique_customers = df['customer_id'].unique()
    np.random.seed(random_state)
    np.random.shuffle(unique_customers)
    
    split_idx = int(len(unique_customers) * (1 - test_size))
    train_customers = unique_customers[:split_idx]
    test_customers = unique_customers[split_idx:]
    
    # Create train/test sets
    train_mask = df['customer_id'].isin(train_customers)
    test_mask = df['customer_id'].isin(test_customers)
    
    X_train = df[train_mask][feature_cols]
    y_train = df[train_mask]['days_to_next_purchase']
    X_test = df[test_mask][feature_cols]
    y_test = df[test_mask]['days_to_next_purchase']
    
    print(f"\nTrain/test split complete (customer-based)")
    print(f"Train customers: {len(train_customers):,}")
    print(f"Test customers: {len(test_customers):,}")
    print(f"Train examples: {len(X_train):,}")
    print(f"Test examples: {len(X_test):,}")
    print(f"Train/test ratio: {len(X_train)/len(X_test):.2f}x")
    
    return X_train, X_test, y_train, y_test

In [7]:
# Model Training, Tuning and Evaluation - Need to Discuss (Container or Seperate Functions)
class NextPurchaseModels:
    """Container for all models with hyperparameter tuning."""
    
    def __init__(self, X_train, y_train, X_test, y_test):
        self.X_train = X_train
        self.y_train = y_train
        self.X_test = X_test
        self.y_test = y_test
        self.all_models = {}
        self.all_metrics = []
    
    def train_linear_regression(self):
        """Linear Regression - Fast baseline."""

        model = LinearRegression()
        model.fit(self.X_train, self.y_train)
        
        y_pred_train = model.predict(self.X_train)
        y_pred_test = model.predict(self.X_test)
        
        mae_train = mean_absolute_error(self.y_train, y_pred_train)
        mae_test = mean_absolute_error(self.y_test, y_pred_test)
        rmse_train = np.sqrt(root_mean_squared_error(self.y_train, y_pred_train))
        rmse_test = np.sqrt(root_mean_squared_error(self.y_test, y_pred_test))
        r2_train = r2_score(self.y_train, y_pred_train)
        r2_test = r2_score(self.y_test, y_pred_test)
        
        print(f"\nModel trained")
        print(f"MAE: {mae_train:.2f} days")
        print(f"RMSE: {rmse_train:.2f} days")
        print(f"R^2: {r2_train:.4f}")
        print(f"\nTest Metrics:")
        print(f"MAE: {mae_test:.2f} days")
        print(f"RMSE: {rmse_test:.2f} days")
        print(f"R^2: {r2_test:.4f}")
        
        # Feature coefficients
        coef_df = pd.DataFrame({
            'feature': self.X_train.columns,
            'coefficient': model.coef_
        }).sort_values('coefficient', ascending=False, key=abs)
        
        print(f"\nTop 10 feature coefficients:")
        print(coef_df.head(10).to_string(index=False))
        
        metrics = {
            'model': 'Linear Regression',
            'hyperparameters': 'None',
            'mae_train': mae_train,
            'mae_test': mae_test,
            'rmse_train': rmse_train,
            'rmse_test': rmse_test,
            'r2_train': r2_train,
            'r2_test': r2_test
        }
        
        self.all_models['Linear Regression'] = model
        self.all_metrics.append(metrics)
        return model, metrics
    
    def train_random_forest_tuned(self):
        """Random Forest with RandomizedSearchCV."""
 
        param_dist = {
            'n_estimators': [50, 100, 200, 300],
            'max_depth': [10, 15, 20, 25, 30, None],
            'min_samples_leaf': [2, 3, 5, 10],
            'min_samples_split': [2, 5, 10],
            'max_features': ['sqrt', 'log2', 0.5, 0.7],
            'bootstrap': [True, False]
        }
        
        print(f"\nHyperparameter space:")
        total_combos = np.prod([len(v) for v in param_dist.values()])
        print(f"Total possible combinations: {total_combos:,}")
        print(f"RandomizedSearchCV will test: 50 iterations")
        
        rf_base = RandomForestRegressor(random_state=42, n_jobs=-1)
        
        random_search = RandomizedSearchCV(
            estimator=rf_base,
            param_distributions=param_dist,
            n_iter=50,
            cv=5,
            scoring='root_mean_squared_error', # Can discuss this
            n_jobs=-1,
            random_state=42,
            verbose=0
        )
        
        print(f"\nTraining RandomizedSearchCV...")
        random_search.fit(self.X_train, self.y_train)
        
        best_model = random_search.best_estimator_
        best_params = random_search.best_params_
        
        print(f"\nBest model found")
        print(f"\nBest hyperparameters:")
        for param, value in sorted(best_params.items()):
            print(f"{param}: {value}")
        print(f"Best CV MAE: {-random_search.best_score_:.2f} days")
        
        # Evaluate
        y_pred_train = best_model.predict(self.X_train)
        y_pred_test = best_model.predict(self.X_test)
        
        mae_train = mean_absolute_error(self.y_train, y_pred_train)
        mae_test = mean_absolute_error(self.y_test, y_pred_test)
        rmse_train = np.sqrt(root_mean_squared_error(self.y_train, y_pred_train))
        rmse_test = np.sqrt(root_mean_squared_error(self.y_test, y_pred_test))
        r2_train = r2_score(self.y_train, y_pred_train)
        r2_test = r2_score(self.y_test, y_pred_test)
        
        print(f"\nModel trained")
        print(f"MAE: {mae_train:.2f} days")
        print(f"RMSE: {rmse_train:.2f} days")
        print(f"R^2: {r2_train:.4f}")
        print(f"\nTest Metrics:")
        print(f"MAE: {mae_test:.2f} days")
        print(f"RMSE: {rmse_test:.2f} days")
        print(f"R^2: {r2_test:.4f}")
        
        # Feature importance
        imp_df = pd.DataFrame({
            'feature': self.X_train.columns,
            'importance': best_model.feature_importances_
        }).sort_values('importance', ascending=False)
        
        print(f"\nTop 10 important features:")
        print(imp_df.head(10).to_string(index=False))
        
        metrics = {
            'model': 'Random Forest (Tuned)',
            'hyperparameters': str(best_params),
            'mae_train': mae_train,
            'mae_test': mae_test,
            'rmse_train': rmse_train,
            'rmse_test': rmse_test,
            'r2_train': r2_train,
            'r2_test': r2_test
        }
        
        self.all_models['Random Forest'] = best_model
        self.all_metrics.append(metrics)
        return best_model, metrics
    
    def train_gradient_boosting_tuned(self):
        """Gradient Boosting with RandomizedSearchCV."""
        
        param_dist = {
            'n_estimators': [100, 200, 300, 500],
            'learning_rate': [0.001, 0.01, 0.05, 0.1, 0.2],
            'max_depth': [3, 4, 5, 6, 7],
            'min_samples_leaf': [1, 2, 3, 5],
            'min_samples_split': [2, 5, 10],
            'subsample': [0.7, 0.8, 0.9, 1.0]
        }
        
        print(f"\nHyperparameter space:")
        total_combos = np.prod([len(v) for v in param_dist.values()])
        print(f"Total possible combinations: {total_combos:,}")
        print(f"RandomizedSearchCV will test: 50 iterations")
        
        gb_base = GradientBoostingRegressor(random_state=42)
        
        random_search = RandomizedSearchCV(
            estimator=gb_base,
            param_distributions=param_dist,
            n_iter=50,
            cv=5,
            scoring='root_mean_squared_error',
            n_jobs=-1,
            random_state=42,
            verbose=0
        )
        
        print(f"\nTraining RandomizedSearchCV...")
        random_search.fit(self.X_train, self.y_train)
        
        best_model = random_search.best_estimator_
        best_params = random_search.best_params_
        
        print(f"\nBest model found")
        print(f"\nBest hyperparameters:")
        for param, value in sorted(best_params.items()):
            print(f"{param}: {value}")
        print(f"Best CV MAE: {-random_search.best_score_:.2f} days")
        
        # Evaluate
        y_pred_train = best_model.predict(self.X_train)
        y_pred_test = best_model.predict(self.X_test)
        
        mae_train = mean_absolute_error(self.y_train, y_pred_train)
        mae_test = mean_absolute_error(self.y_test, y_pred_test)
        rmse_train = np.sqrt(root_mean_squared_error(self.y_train, y_pred_train))
        rmse_test = np.sqrt(root_mean_squared_error(self.y_test, y_pred_test))
        r2_train = r2_score(self.y_train, y_pred_train)
        r2_test = r2_score(self.y_test, y_pred_test)
        
        print(f"\nModel trained")
        print(f"MAE: {mae_train:.2f} days")
        print(f"RMSE: {rmse_train:.2f} days")
        print(f"R^2: {r2_train:.4f}")
        print(f"\nTest Metrics:")
        print(f"MAE: {mae_test:.2f} days")
        print(f"RMSE: {rmse_test:.2f} days")
        print(f"R^2: {r2_test:.4f}")
        
        # Feature importance
        imp_df = pd.DataFrame({
            'feature': self.X_train.columns,
            'importance': best_model.feature_importances_
        }).sort_values('importance', ascending=False)
        
        print(f"\nTop 10 important features:")
        print(imp_df.head(10).to_string(index=False))
        
        metrics = {
            'model': 'Gradient Boosting (Tuned)',
            'hyperparameters': str(best_params),
            'mae_train': mae_train,
            'mae_test': mae_test,
            'rmse_train': rmse_train,
            'rmse_test': rmse_test,
            'r2_train': r2_train,
            'r2_test': r2_test
        }
        
        self.all_models['Gradient Boosting'] = best_model
        self.all_metrics.append(metrics)
        return best_model, metrics
    
    def train_xgboost_tuned(self):
        """XGBoost with RandomizedSearchCV."""
        
        param_dist = {
            'n_estimators': [100, 200, 300, 500],
            'learning_rate': [0.001, 0.01, 0.05, 0.1],
            'max_depth': [3, 4, 5, 6, 7],
            'min_child_weight': [1, 3, 5, 7],
            'subsample': [0.7, 0.8, 0.9, 1.0],
            'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
            'gamma': [0, 0.1, 0.5, 1.0]
        }
        
        print(f"\nHyperparameter space:")
        total_combos = np.prod([len(v) for v in param_dist.values()])
        print(f"Total possible combinations: {total_combos:,}")
        print(f"RandomizedSearchCV will test: 50 iterations")
        
        xgb_base = xgb.XGBRegressor(random_state=42, n_jobs=-1)
        
        random_search = RandomizedSearchCV(
            estimator=xgb_base,
            param_distributions=param_dist,
            n_iter=50,
            cv=5,
            scoring='root_mean_squared_error',
            n_jobs=-1,
            random_state=42,
            verbose=0
        )
        
        print(f"\nTraining RandomizedSearchCV...")
        random_search.fit(self.X_train, self.y_train)
        
        best_model = random_search.best_estimator_
        best_params = random_search.best_params_
        
        print(f"\nBest model found")
        print(f"\nBest hyperparameters:")
        for param, value in sorted(best_params.items()):
            print(f"{param}: {value}")
        print(f"Best CV MAE: {-random_search.best_score_:.2f} days")
        
        # Evaluate
        y_pred_train = best_model.predict(self.X_train)
        y_pred_test = best_model.predict(self.X_test)
        
        mae_train = mean_absolute_error(self.y_train, y_pred_train)
        mae_test = mean_absolute_error(self.y_test, y_pred_test)
        rmse_train = np.sqrt(root_mean_squared_error(self.y_train, y_pred_train))
        rmse_test = np.sqrt(root_mean_squared_error(self.y_test, y_pred_test))
        r2_train = r2_score(self.y_train, y_pred_train)
        r2_test = r2_score(self.y_test, y_pred_test)
        
        print(f"\nModel trained")
        print(f"MAE: {mae_train:.2f} days")
        print(f"RMSE: {rmse_train:.2f} days")
        print(f"R^2: {r2_train:.4f}")
        print(f"\nTest Metrics:")
        print(f"MAE: {mae_test:.2f} days")
        print(f"RMSE: {rmse_test:.2f} days")
        print(f"R^2: {r2_test:.4f}")
        
        # Feature importance
        imp_df = pd.DataFrame({
            'feature': self.X_train.columns,
            'importance': best_model.feature_importances_
        }).sort_values('importance', ascending=False)
        
        print(f"\nTop 10 important features:")
        print(imp_df.head(10).to_string(index=False))
        
        metrics = {
            'model': 'XGBoost (Tuned)',
            'hyperparameters': str(best_params),
            'mae_train': mae_train,
            'mae_test': mae_test,
            'rmse_train': rmse_train,
            'rmse_test': rmse_test,
            'r2_train': r2_train,
            'r2_test': r2_test
        }
        
        self.all_models['XGBoost'] = best_model
        self.all_metrics.append(metrics)
        return best_model, metrics
    
    def compare_all_models(self):
        """Compare all trained models."""
        
        comparison_df = pd.DataFrame(self.all_metrics).sort_values('rmse_test')
        
        # Add ranking
        comparison_df['RMSE_Rank'] = range(1, len(comparison_df) + 1)
        comparison_df['R2_Rank'] = comparison_df['r2_test'].rank(ascending=False).astype(int)
        
        print("\n")
        print(comparison_df[[
            'model', 'mae_test', 'rmse_test', 'r2_test', 'MAE_Rank', 'R2_Rank'
        ]].to_string(index=False))
        
        best_rmse = comparison_df.loc[comparison_df['rmse_test'].idxmin()]
        best_r2 = comparison_df.loc[comparison_df['r2_test'].idxmax()]
        
        print(f"\nBest RMSE (lowest prediction error): {best_rmse['model']}")
        print(f"RMSE: {best_rmse['mae_test']:.2f} days")
        print(f"R^2: {best_rmse['r2_test']:.4f}")
        
        print(f"\nBest R^2 (best fit): {best_r2['model']}")
        print(f"RMSE: {best_r2['rmse_test']:.2f} days")
        print(f"R²: {best_r2['r2_test']:.4f}")
        
        # Visualization
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        
        axes[0].barh(comparison_df['model'], comparison_df['mae_test'], color='steelblue')
        axes[0].set_xlabel('MAE (Days)', fontsize=11)
        axes[0].set_title('Mean Absolute Error', fontsize=12, fontweight='bold')
        axes[0].invert_yaxis()
        for i, v in enumerate(comparison_df['mae_test']):
            axes[0].text(v + 0.1, i, f'{v:.2f}', va='center', fontsize=10)
        
        axes[1].barh(comparison_df['model'], comparison_df['rmse_test'], color='coral')
        axes[1].set_xlabel('RMSE (Days)', fontsize=11)
        axes[1].set_title('Root Mean Squared Error', fontsize=12, fontweight='bold')
        axes[1].invert_yaxis()
        for i, v in enumerate(comparison_df['rmse_test']):
            axes[1].text(v + 0.1, i, f'{v:.2f}', va='center', fontsize=10)
        
        axes[2].barh(comparison_df['model'], comparison_df['r2_test'], color='lightgreen')
        axes[2].set_xlabel('R^2 Score', fontsize=11)
        axes[2].set_title('R^2 Score', fontsize=12, fontweight='bold')
        axes[2].invert_yaxis()
        for i, v in enumerate(comparison_df['r2_test']):
            axes[2].text(v + 0.01, i, f'{v:.4f}', va='center', fontsize=10)
        
        plt.tight_layout()
        plt.savefig('model_comparison_results.png', dpi=300, bbox_inches='tight')
        print(f"\nComparison chart saved as 'model_comparison_results.png'")
        
        return comparison_df
    
    def train_all_models(self):
        """Train all models in sequence."""
        
        print("\nTraining Linear Regression...")
        self.train_linear_regression()
        
        print("\nTraining Random Forest...")
        self.train_random_forest_tuned()
        
        print("\nTraining Gradient Boosting...")
        self.train_gradient_boosting_tuned()
        
        print("\nTraining XGBoost...")
        self.train_xgboost_tuned()
        
        return self.all_models

In [8]:
# Inference
def predict_next_purchase_for_customer(customer_id, df, rfm_context, best_model, scaler, le_category, feature_cols):
    """
    Predict next purchase for a specific customer.
    """
    
    # Get customer's transaction history
    customer_txns = df[df['customer_id'] == customer_id].sort_values('purchase_date')
    
    if len(customer_txns) == 0:
        print(f"Customer {customer_id} not found in data")
        return None
    
    # Get latest transaction
    latest_txn_date = customer_txns['purchase_date'].max()
    
    # Historical features
    hist_frequency = len(customer_txns['order_id'].unique())
    hist_monetary = customer_txns['line_total'].sum()
    hist_avg_order = hist_monetary / hist_frequency if hist_frequency > 0 else 0
    hist_days_active = (customer_txns['purchase_date'].max() - 
                        customer_txns['purchase_date'].min()).days
    
    category_counts = customer_txns['category'].value_counts()
    top_category = category_counts.idxmax()
    category_concentration = category_counts.max() / len(customer_txns)
    
    # Get customer context
    customer_context = rfm_context[rfm_context['customer_id'] == customer_id]
    
    if customer_context.empty:
        print(f"Customer {customer_id} not found in RFM context")
        return None
    
    # Build feature vector
    top_category_encoded = le_category.transform([top_category])[0]
    
    feature_dict = {
        'hist_frequency': hist_frequency,
        'hist_monetary': hist_monetary,
        'hist_avg_order': hist_avg_order,
        'hist_days_active': hist_days_active,
        'category_concentration': category_concentration,
        'segment_encoded': customer_context['segment_encoded'].values[0],
        'kmeans_cluster': customer_context['kmeans_cluster'].values[0],
        'monetary': customer_context['monetary'].values[0],
        'recency_days': customer_context['recency_days'].values[0],
        'frequency': customer_context['frequency'].values[0],
        'top_category_encoded': top_category_encoded
    }
    
    X_customer = pd.DataFrame([feature_dict])[feature_cols]
    
    # Scale numerical features
    numerical_cols = [col for col in feature_cols if col != 'segment_encoded' 
                      and col != 'kmeans_cluster' and col != 'top_category_encoded']
    X_customer_scaled = X_customer.copy()
    X_customer_scaled[numerical_cols] = scaler.transform(X_customer[numerical_cols])
    
    # Make prediction
    days_to_next = best_model.predict(X_customer_scaled)[0]
    predicted_date = latest_txn_date + timedelta(days=int(days_to_next))
    
    return {
        'customer_id': customer_id,
        'latest_purchase': latest_txn_date.strftime('%Y-%m-%d'),
        'predicted_days_to_next': int(days_to_next),
        'predicted_purchase_date': predicted_date.strftime('%Y-%m-%d'),
        'predicted_category': top_category,
        'customer_segment': customer_context['segment'].values[0],
        'lifetime_value': customer_context['monetary'].values[0]
    }

In [12]:
# Load and merge data
df_merged = load_and_merge_data(
    customers_path='data/clean_customers.csv',
    orders_path='data/clean_orders.csv',
    products_path='data/clean_products.csv'
)

print("\nData loaded and merged successfully!")
print(f"\nDataset shape: {df_merged.shape}")
print(f"Columns: {df_merged.columns.tolist()}")
print(df_merged.head())


Loading datasets
Customers: 18,070 rows
Orders: 97,702 rows (may include multiple items per order)
Products: 14,784 rows

Merging orders with product categories
Orders + Products merged: 97,702 rows

Aggregating multiple items per order...
Original rows (with multiple items per order): 97,702
Aggregated to unique orders: 51,216

Merging with customer metrics...
Final merged dataset: 51,216 rows

Dataset ready for pipeline:
Columns: ['customer_id', 'order_id', 'purchase_date', 'line_total', 'category', 'quantity', 'tenure_bucket', 'total_orders', 'has_returned']
Date range: 2024-01-01 to 2026-02-18
Total customers: 32,663
Total unique orders: 51,216
Average items per order: 3.56

Data loaded and merged successfully!

Dataset shape: (51216, 9)
Columns: ['customer_id', 'order_id', 'purchase_date', 'line_total', 'category', 'quantity', 'tenure_bucket', 'total_orders', 'has_returned']
  customer_id      order_id purchase_date  line_total   category  quantity  \
0      T10002   TS271275504 

In [13]:
# Check for any missing values
print(f"\nMissing values:")
print(df_merged.isnull().sum())

# Fix Missing Values


Missing values:
customer_id          0
order_id             0
purchase_date        0
line_total           0
category           696
quantity             0
tenure_bucket    15627
total_orders     15627
has_returned     15627
dtype: int64


In [14]:
# Load RFM Data
rfm_registered = pd.read_csv('rfm_with_kmeans_context.csv')

print(f"RFM data loaded: {len(rfm_registered):,} customers")
print(f"Columns: {rfm_registered.columns.tolist()}")
print(f"\nFirst few rows:")
print(rfm_registered[['customer_id', 'segment', 'kmeans_cluster', 'monetary', 'recency_days', 'frequency']].head())

# Verify all customers from df_merged are in RFM
missing_customers = set(df_merged['customer_id'].unique()) - set(rfm_registered['customer_id'].unique())
if len(missing_customers) > 0:
    print(f"\nWarning: {len(missing_customers)} customers not in RFM data")
else:
    print(f"\nAll {df_merged['customer_id'].nunique():,} customers in RFM data")

RFM data loaded: 19,152 customers
Columns: ['customer_id', 'recency_days', 'frequency', 'monetary', 'avg_order_value', 'first_purchase', 'last_purchase', 'active_days', 'R_score', 'F_score', 'M_score', 'RFM_score', 'segment', 'kmeans_cluster', 'primary_category', 'context_segment', 'segment_id', 'cluster_id', 'category_id']

First few rows:
  customer_id              segment  kmeans_cluster  monetary  recency_days  \
0      T10002          Hibernating               0   2170.00           419   
1      T10008            Champions               0   7992.35           174   
2       T1002  Potential Loyalists               0   2366.00            82   
3      T10060          Hibernating               0   3508.80           539   
4      T10061            Champions               0  42760.79            31   

   frequency  
0          1  
1          4  
2          1  
3          1  
4         22  



In [ ]:
# Training Data Preparation
training_df = prepare_next_purchase_data(
    df=df_merged,
    rfm_context=rfm_registered,
    min_purchases=2
)

# Display the result
print(f"\nTraining data prepared: {len(training_df):,} training examples")
print(f"\nTraining data structure:")
print(training_df.head())

# Visualize the distribution
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(training_df['days_to_next_purchase'], bins=50, color='steelblue', edgecolor='black')
plt.xlabel('Days to Next Purchase')
plt.ylabel('Frequency')
plt.title('Distribution of Days Until Next Purchase')
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
training_df['next_category'].value_counts().plot(kind='barh', color='coral')
plt.xlabel('Count')
plt.title('Next Purchase Category Distribution')
plt.grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

In [ ]:
# Feature Engineering
df_engineered, feature_cols, scaler, le_segment, le_category = engineer_features(training_df)

print(f"\nFeatures engineered")
print(f"Total features: {len(feature_cols)}")
print(f"Features: {feature_cols}")

# Show feature statistics
print(f"\nFeature statistics:")
print(df_engineered[feature_cols].describe())
# Show encoded values
print(f"\nUnique values in encoded features:")
print(f"Segments: {df_engineered['segment_encoded'].nunique()}")
print(f"Clusters: {df_engineered['kmeans_cluster'].nunique()}")
print(f"Categories: {df_engineered['top_category_encoded'].nunique()}")

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = split_data_by_customer(
    df=df_engineered,
    feature_cols=feature_cols,
    test_size=0.2,
    random_state=42
)

print(f"\nTrain/test split complete")
print(f"\nTrain set:")
print(f"Customers: {df_engineered[df_engineered.index.isin(X_train.index)]['customer_id'].nunique():,}")
print(f"Examples: {len(X_train):,}")

print(f"\nTest set:")
print(f"Customers: {df_engineered[df_engineered.index.isin(X_test.index)]['customer_id'].nunique():,}")
print(f"Examples: {len(X_test):,}")

In [ ]:
# Intialize trainer with training data
trainer = NextPurchaseModels(X_train, y_train, X_test, y_test)

In [ ]:
# Train linear regression model
lr_model, lr_metrics = trainer.train_linear_regression()

In [ ]:
# Train random forest model
rf_model, rf_metrics = trainer.train_random_forest_tuned()

In [ ]:
# Train gradient boosting model
gb_model, gb_metrics = trainer.train_gradient_boosting_tuned()

In [ ]:
# Train XGBoost model
xgb_model, xgb_metrics = trainer.train_xgboost_tuned()

In [ ]:
# Compare all models
comparison_df = trainer.compare_all_models()

# Display as nice table
print("\nModel Comparison Results:")
print(comparison_df[['model', 'mae_test', 'rmse_test', 'r2_test']].to_string(index=False))

# Find best model
best_model_name = comparison_df.iloc[0]['model']
best_rmse = comparison_df.iloc[0]['rmse_test']

print(f"\nBest Model: {best_model_name}")
print(f"RMSE: {best_rmse:.2f} days")

In [ ]:
# Make Prediction on Individual Customers

# Get best model
best_model_obj = trainer.all_models[comparison_df.iloc[0]['model'].replace(' (Tuned)', '')]

# Select a few high-value customers to predict for
high_value_customers = rfm_registered.nlargest(5, 'monetary')['customer_id'].tolist()

print(f"\nMaking predictions for top {len(high_value_customers)} customers by lifetime value:\n")

predictions_list = []

for customer_id in high_value_customers:
    prediction = predict_next_purchase_for_customer(
        customer_id=customer_id,
        df=df_merged,
        rfm_context=rfm_registered,
        best_model=best_model_obj,
        scaler=scaler,
        le_category=le_category,
        feature_cols=feature_cols
    )
    
    if prediction:
        predictions_list.append(prediction)
        print(f"Customer {prediction['customer_id']}")
        print(f"Last purchase: {prediction['latest_purchase']}")
        print(f"Will buy in ~{prediction['predicted_days_to_next']} days")
        print(f"Expected date: {prediction['predicted_purchase_date']}")
        print(f"Expected category: {prediction['predicted_category']}")
        print(f"Segment: {prediction['customer_segment']}")
        print(f"Lifetime value: ${prediction['lifetime_value']:,.2f}")
        print()

# Create DataFrame of predictions
predictions_df = pd.DataFrame(predictions_list)
print("\nPredictions Summary:")
print(predictions_df.to_string(index=False))